In [ ]:
import os
import pandas as pd
import json
import oci
from oci.auth import signers

In [8]:
# Save to JSON file
def csv_to_json(file_path, output_path, drop_columns=None):
    """
    Load CSV and convert to JSON records (list of dicts).

    Args:
        file_path (str): Path to CSV file.
        drop_columns (list): List of column names to drop.

    Returns:
        list: JSON-like records (list of dicts).
    """
    df = pd.read_csv(file_path)

    if drop_columns:
        df = df.drop(columns=drop_columns, errors="ignore")

    return df.to_json(output_path, orient="records", lines=False, indent=2)

folder_path = "preprocessed_data"
output_dir = "injestion_data"

# To keep text column only
# drop_columns = ["character_count", "status", "text_processed", "text_no_stopwords", "text_stemmed", "text_lemmatized"]

# To keep text_processed column only
# drop_columns = ["character_count", "status", "text", "text_no_stopwords", "text_stemmed", "text_lemmatized"]

# To keep text_lemmatized column only
# drop_columns = ["character_count", "status", "text", "text_processed", "text_no_stopwords", "text_stemmed"]

# To keep text and text_lemmatized columns only
drop_columns = ["character_count", "status", "text_processed", "text_no_stopwords", "text_stemmed"]

for filename in os.listdir(folder_path):
    print(f"Processing file: {filename}")
    if filename.endswith(".csv"):
        file_path = os.path.join(folder_path, filename)
        output_path = os.path.join(output_dir, os.path.splitext(filename)[0] + ".json")
        #base_name = os.path.splitext(filename)[0]

        csv_to_json(file_path, output_path, drop_columns)

Processing file: oracle_docs_preprocessed.csv


In [ ]:
signer = signers.get_resource_principals_signer()
object_storage_client = oci.object_storage.ObjectStorageClient(config={}, signer=signer)
namespace = object_storage_client.get_namespace().data
bucket_name = 'oci-rag-bucket'
for filename in os.listdir("injestion_data"):
    with open(filename, 'rb') as f:
        object_storage_client.put_object(
            namespace,
            bucket_name,
            filename,
            f
        )